In [16]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    'sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'Temperature': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Strong', 'Weak'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
})

print(data)

# Attributes and Target
attributes = ['sky', 'Temperature', 'Humidity', 'Wind']
target = 'EnjoySport'

print("Attributes:", attributes)
print("Target:", target)

S = ['@'] * len(attributes)
G = [['?'] * len(attributes)]

print("Initial S:", S)
print("Initial G:", G)


# Covers function
def covers(hypothesis, example):
    for h, e in zip(hypothesis, example):
        if h != '?' and h != '@' and h != e:
            return False
    return True


# Generalize S
def generalize_S(S, example):
    new_S = S.copy()

    for i in range(len(S)):
        if S[i] == '@':
            new_S[i] = example[i]
        elif S[i] != example[i]:
            new_S[i] = '?'

    return new_S


# Possible values
domains = [
    ['Sunny', 'Rainy'],
    ['Warm', 'Cold'],
    ['Normal', 'High'],
    ['Strong', 'Weak']
]


# Specialize G
def specialize_G(G, example, S, domains):
    new_G = []

    for g in G:

        # Check whether G covers the negative example
        if covers(g, example):

            # Try every attribute
            for i in range(len(g)):

                if g[i] == '?':

                    # Try possible values
                    for value in domains[i]:

                        # Values must reject the negative example
                        if value != example[i]:

                            hypothesis = g.copy()
                            hypothesis[i] = value

                            # Check consistency with S
                            valid = True

                            for j in range(len(S)):
                                if S[j] != '@' and hypothesis[j] != '?':
                                    if hypothesis[j] != S[j]:
                                        valid = False
                                        break

                            if valid:
                                new_G.append(hypothesis)

        else:
            new_G.append(g)

    return new_G


# Candidate Elimination
for index, row in data.iterrows():

    example = row[attributes].tolist()
    label = row[target]

    print("\n======================")
    print("Example", index + 1)
    print("Example", example)
    print("Target", label)

    # Positive example
    if label == 'Yes':

        print("Positive example")

        # Remove G hypotheses that don't cover positive example
        G = [g for g in G if covers(g, example)]

        # Generalize S
        S = generalize_S(S, example)

    # Negative example
    else:

        print("Negative example")

        # Specialize G
        G = specialize_G(G, example, S, domains)

    print("S =", S)
    print("G =", G)

     sky Temperature Humidity    Wind EnjoySport
0  Sunny        Warm   Normal  Strong        Yes
1  Sunny        Warm     High  Strong        Yes
2  Rainy        Cold     High  Strong         No
3  Sunny        Warm     High    Weak        Yes
Attributes: ['sky', 'Temperature', 'Humidity', 'Wind']
Target: EnjoySport
Initial S: ['@', '@', '@', '@']
Initial G: [['?', '?', '?', '?']]

Example 1
Example ['Sunny', 'Warm', 'Normal', 'Strong']
Target Yes
Positive example
S = ['Sunny', 'Warm', 'Normal', 'Strong']
G = [['?', '?', '?', '?']]

Example 2
Example ['Sunny', 'Warm', 'High', 'Strong']
Target Yes
Positive example
S = ['Sunny', 'Warm', '?', 'Strong']
G = [['?', '?', '?', '?']]

Example 3
Example ['Rainy', 'Cold', 'High', 'Strong']
Target No
Negative example
S = ['Sunny', 'Warm', '?', 'Strong']
G = [['Sunny', '?', '?', '?'], ['?', 'Warm', '?', '?']]

Example 4
Example ['Sunny', 'Warm', 'High', 'Weak']
Target Yes
Positive example
S = ['Sunny', 'Warm', '?', '?']
G = [['Sunny', '?', '?', '